# QuickTalk: train a small model from scratch (Colab, A100)

Trains a GPT-style model on `train/build/` from the `sraivante/quicktalk` repo: first on plain text (books + behaviour
passages), then on chats (behaviour Q&A + 13 conversation patterns).

**Before you start**
1. *Runtime > Change runtime type > A100 GPU*. The notebook stops if the GPU is not an A100.
2. The repo is public, so no token is needed. (Only if it is made private later: add a Colab secret named
   `GITHUB_TOKEN` with a GitHub token that can read the repo.)
3. *Runtime > Run all*.

**Crashes, disconnects and idle timeouts**
- Everything is saved to Google Drive (`MyDrive/quicktalk_run`): tokenizer, token files, checkpoints every 5 minutes,
  logs. If the runtime is killed, reconnect and *Run all* again: finished steps are skipped and training resumes from
  the last checkpoint.
- Training runs in the background, so closing the browser tab does not stop it while the runtime is alive. With Colab
  Pro+ you can also turn on background execution so the runtime keeps running after you close the tab.
- The whole run is short (see the plan cell for the step count), so a session timeout is unlikely to hit it.

**Model size:** the plan cell uses your rule of 10 training tokens per parameter (not Chinchilla's 20).

**Runs:** each run lives in its own Drive folder (`RUN` in cell 2), so earlier models are kept.
Full history and results: `train/colab/colab_fixes.md` in the repo.
- Runs 1-4 (8.9M-12.6M parameters, your data only): learned reply formats but not meaning (too little text).
- Run 5 `quicktalk_run5` (69.5M parameters): added ~0.6B tokens of simple English (TinyStories + Simple English
  Wikipedia) for pretraining; chat training on your data only. Offline test: 22% overall (`train/test_run5/report.md`).
- Run 6 `quicktalk_run6` (69.5M, continued from run 5 on +1.3B tokens incl. FineWeb-Edu, SODA, WordNet; 13 chat
  patterns x 1,200): offline test 28% (`train/test_run6/report.md`).
- Run 7 `quicktalk_run7` (97.6M parameters, ~2B tokens, run 5/6 tokenizer): offline test 29%; maths weak because
  the tokenizer merged digits (`train/test_run7/report.md`, `train/test_math/`).
- **Run 8 `quicktalk_run8` (this version): the same ~100M shape (97.6M: d 768, 12 layers, 12 heads), trained from
  scratch with a NEW tokenizer that splits numbers into single digits, a 1,024-token context (was 512) and ~10B
  pretraining tokens (5x run 7).**
  - Pretraining: your text 3x + TinyStories + Simple English Wikipedia + WordNet (3x) + SODA dialogues + 10 shards of
    FineWeb-Edu sample-10BT (~7-9B tokens). One pass, learning rate 6e-4.
  - Chat training: all earlier chat data plus run 8: messy/misspelled questions, JSON answers, answering from a given
    text (or own knowledge, or a polite "I don't know"), greetings, short polite "I don't know", passage questions,
    multi-turn chats, stories. Conversation patterns 2x, behaviour capped at 25,000, 2 passes; the best checkpoint by
    chat eval loss is kept (no over-training).
  - Drive space: ~35 GB of downloaded text (`MyDrive/quicktalk_extra8`) + ~22 GB of token files + ~4 GB checkpoints.
- Expected time on the A100: downloads ~1-2 h, tokenizing ~2-4 h, pretraining ~11-14 h, chat training ~15 min.
  Every step resumes: if Colab disconnects, reconnect and *Run all*. Finished download shards and token parts are
  skipped; training continues from the last checkpoint (saved every 15 minutes).


In [ ]:
# 1. Check the GPU: A100 only
import subprocess, torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > A100 GPU'
name = torch.cuda.get_device_name(0)
print('GPU:', name)
assert 'A100' in name, f'This runtime has {name}, not an A100. Change the runtime type to A100 and run again.'

In [ ]:
# 2. Google Drive holds all state, so nothing is lost if the runtime restarts
import os
RUN = 'quicktalk_run8'          # this experiment's folder (new tokenizer, so nothing is copied from older runs)
EXTRA_SETS = 'tinystories,simplewiki,wordnet,soda,fineweb_files'
FINEWEB_FILES = 10              # FineWeb-Edu sample-10BT shards (~0.7-0.9B tokens each)
OWN_REPEAT, CHAT_REPEAT = 3, 2  # your pretraining text 3x; conversation patterns 2x in chat training
CAPS = '--cap behaviour=25000'  # chat training: at most 25,000 behaviour Q&A (the rest are patterns + maths)
PRE_EPOCHS, SFT_EPOCHS = 1, 2   # passes over pretraining text / chats
SHAPE, BLOCK = '768,12,12', 1024  # ~100M model (97.6M parameters); 1,024-token context
PRE_LR = 6e-4
if not os.path.isdir('/content/drive/MyDrive'):
    from google.colab import drive
    try:
        drive.mount('/content/drive', force_remount=True, timeout_ms=300000)
    except Exception as e:
        print('Drive mount failed:', e)
        print("""Fix, then run this cell again:
 1. In the Google sign-in pop-up, choose the SAME account you use for Colab, and on the permissions page tick
    'Select all' (full Drive access). Unticking any box makes the mount fail.
 2. Allow pop-ups and third-party cookies for colab.research.google.com.
 3. Or click the folder icon on the left, then the 'Mount Drive' icon, and run this cell again.
 4. Still failing: Runtime > Disconnect and delete runtime, reconnect (A100), Run all.""")
        raise
BASE = '/content/drive/MyDrive'
WORK = f'{BASE}/{RUN}'
EXTRA = f'{BASE}/quicktalk_extra8'   # downloaded text on Drive (~35 GB), so a restart never downloads it again
os.makedirs(WORK, exist_ok=True); os.makedirs(EXTRA, exist_ok=True)
import shutil
free = shutil.disk_usage(BASE).free / 1e9
print('work dir:', WORK, sorted(os.listdir(WORK)), f'| Drive free: {free:,.0f} GB')
if free < 70: print('WARNING: run 8 needs ~70 GB free on Google Drive (downloads + token files + checkpoints).')

In [ ]:
# 3. Get the code and the built data (sparse clone: only train/build and train/colab)
import os, subprocess
REPO, BRANCH, SRC = 'sraivante/quicktalk', 'claude/intelligent-ride-oxqjfy', '/content/quicktalk'
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    token = None
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists(f'{SRC}/train/build/manifest.json'):
    subprocess.run(['rm', '-rf', SRC])
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, '--filter=blob:none', '--sparse', url, SRC], check=True)
    subprocess.run(['git', '-C', SRC, 'sparse-checkout', 'set', 'train/build', 'train/colab'], check=True)
else:                                    # already cloned in this runtime: update to the latest code and data
    subprocess.run(['git', '-C', SRC, 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', SRC, 'reset', '--hard', 'FETCH_HEAD'], check=True)
print(sorted(os.listdir(f'{SRC}/train/build')))
LM = f'{SRC}/train/colab/quicktalk_lm.py'
DATA = f'{SRC}/train/build'

In [ ]:
# 4. Libraries, and the Hugging Face token (needed: ~30 GB of FineWeb-Edu downloads)
#    Add it once as a Colab secret: key icon on the left > Add new secret > Name HF_TOKEN, Value hf_..., turn on
#    "Notebook access". Never paste the token into this notebook (the notebook and the repo are shareable).
!pip -q install tokenizers datasets huggingface_hub nltk pyarrow
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN'); print('HF_TOKEN found: downloads use your account')
except Exception:                        # no secret: ask once (typed into a hidden box, never saved in the notebook)
    from getpass import getpass
    tok = getpass('Paste your Hugging Face token and press Enter (or just Enter to skip): ').strip()
    if tok: os.environ['HF_TOKEN'] = tok; print('HF_TOKEN set for this session')
    else: print('no HF_TOKEN: downloads are anonymous (slower, may be rate-limited)')

In [ ]:
# 5. Download text, train the digit tokenizer, token files, plan (every step skips what is already done)
#    Safe to stop and re-run: finished FineWeb shards and finished token parts are kept.
import os, json
if os.path.exists(f'{WORK}/plan.json'):          # guard against a token set from an interrupted download
    n = json.load(open(f'{WORK}/plan.json'))['pretrain_tokens']
    assert n > 5e9, (f'{WORK} holds a small pretraining set ({n:,} tokens, expected ~8-10B). Delete {WORK}/plan.json '
                     f'and {WORK}/data, then Run all (downloads in {EXTRA} are kept).')
if not os.path.exists(f'{WORK}/plan.json'):
    !python {LM} fetch --out {EXTRA} --sets {EXTRA_SETS} --fineweb-files {FINEWEB_FILES} --tmp /content/hf_tmp
    have = [f for f in os.listdir(EXTRA) if f.startswith('fineweb10bt_') and f.endswith('.txt')]
    missing = [n for n in EXTRA_SETS.split(',') if n != 'fineweb_files' and not os.path.exists(f'{EXTRA}/{n}.txt')]
    assert not missing and len(have) >= FINEWEB_FILES, f'download incomplete ({missing}, {len(have)} shards): re-run this cell'
!python {LM} tokenizer --data {DATA} --workdir {WORK} --extra {EXTRA} --extra-sample-mb 80 --digits
!python {LM} prepare --data {DATA} --workdir {WORK} --extra {EXTRA} --own-repeat {OWN_REPEAT} --chat-repeat {CHAT_REPEAT} --extra-repeat wordnet.txt=3 --parts {CAPS}
!python {LM} plan --workdir {WORK} --pretrain-epochs {PRE_EPOCHS} --sft-epochs {SFT_EPOCHS} --dropout 0 --shape {SHAPE} --block {BLOCK}
pl = json.load(open(f'{WORK}/plan.json'))
print(f"pretraining tokens: {pl['pretrain_tokens'] / 1e9:.2f}B -> about {pl['pretrain_tokens'] / 230e3 / 3600:.0f} h on the A100")

In [ ]:
# 6. Start training in the background (pretrain from scratch, then chat fine-tuning). Safe to re-run: it resumes.
import os, subprocess
os.makedirs('/content/cache', exist_ok=True)
print('copying token files to the local disk (~22 GB, a few minutes)...')
subprocess.run(f'rsync -a {WORK}/data/ /content/cache/data/', shell=True, check=True)   # local copy reads faster than Drive
LOG = f'{WORK}/train.log'
cmd = (f'python {LM} train --workdir {WORK} --data-dir /content/cache/data --stage pretrain --batch 64 '
       f'--ckpt-every 2000 --ckpt-minutes 15 --eval-every 1000 --lr {PRE_LR} && '
       f'python {LM} train --workdir {WORK} --data-dir /content/cache/data --stage sft --batch 64 --eval-every 100 --keep-best')
running = subprocess.run("pgrep -f '[q]uicktalk_lm.py train'", shell=True, capture_output=True).stdout.strip()
if running:
    print('training is already running; watch it in the next cell')
else:
    subprocess.Popen(f'nohup bash -c "{cmd}" >> {LOG} 2>&1 &', shell=True)
    print('started; log:', LOG)

In [ ]:
# 7. Watch progress (re-run this cell any time; stop it with the stop button, training keeps going)
import time, subprocess, os
LOG = f'{WORK}/train.log'
while True:
    out = subprocess.run(f'tail -n 6 {LOG}', shell=True, capture_output=True, text=True).stdout
    print(out, flush=True)
    done = os.path.exists(f'{WORK}/checkpoints/sft_final.pt')
    alive = subprocess.run("pgrep -f '[q]uicktalk_lm.py train'", shell=True, capture_output=True).stdout.strip()
    if done: print('DONE: model at', f'{WORK}/checkpoints/sft_final.pt'); break
    if not alive: print('training is not running: check the log above, then re-run cell 6 to resume'); break
    time.sleep(60)

In [ ]:
# 8. Loss curves (step, train loss, eval loss)
import pandas as pd, matplotlib.pyplot as plt, os
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for i, st in enumerate(['pretrain', 'sft']):
    p = f'{WORK}/log_{st}.csv'
    if os.path.exists(p):
        d = pd.read_csv(p, names=['step', 'train_loss', 'eval_loss'])
        ax[i].plot(d['step'], d['train_loss'], label='train')
        ax[i].plot(d['step'], d['eval_loss'], label='eval')
        ax[i].set_title(st); ax[i].legend()
plt.show()

In [ ]:
# 9. Talk to the model: every run gets the same questions, each in a fresh conversation
import os
PROMPTS = ' '.join(f'--prompt "{q}"' for q in [
    'Hi! How was your day?',
    'What should I do if my friend is upset with me?',
    'Correct this: She dont like apples.',
    'What does the word diligent mean?',
    'Kal mera exam hai, I am very nervous. What should I do?',
    'Write a short note to thank my teacher.',
    '1. What is a synonym for happy? 2. Correct this: He go to school yesterday.',
    'What will the weather be in my town tomorrow?',
    'I have 3 boxes with 12 pencils in each. How many pencils do I have?',
    'Rewrite in the past tense: I walk to school and meet my friends.',
    'i wat to lern swiming but i am scard of watr what shud i do',
    'Give me this as JSON with keys name and city: Meera lives in Pune.',
    'Note: The library is closed on Sunday and opens at 9 am on Monday. Question: When does the library open on Monday?',
    'Good morning! How are you today?',
])
for run in sorted(os.listdir(BASE)):
    if run.startswith('quicktalk_run') and os.path.exists(f'{BASE}/{run}/checkpoints/sft_final.pt'):
        print(f'\n===== {run} =====')
        !python {LM} chat --workdir {BASE}/{run} {PROMPTS} --temperature 0.6

In [ ]:
# 10. Compare runs: model size and final eval loss (lower is better; runs 5+ use a new tokenizer and run 6 a new eval split, so compare those by the chat test)
import json, os, pandas as pd
rows = []
for run in sorted(os.listdir(BASE)):
    w = f'{BASE}/{run}'
    if run.startswith('quicktalk_run') and os.path.exists(f'{w}/plan.json'):
        pl = json.load(open(f'{w}/plan.json')); r = {'run': run, 'params_M': round(pl['params'] / 1e6, 1),
             'passes': f"{pl['pretrain_epochs']}+{pl['sft_epochs']}", 'chat_repeat': pl.get('chat_repeat', 1)}
        for st in ('pretrain', 'sft'):
            p = f'{w}/log_{st}.csv'
            if os.path.exists(p): r[f'{st}_eval_loss'] = pd.read_csv(p, names=['s', 't', 'e'])['e'].iloc[-1]
        rows.append(r)
pd.DataFrame(rows)

In [ ]:
# 11. Finish: wait for the model, zip it to Drive, verify the zip, then disconnect this runtime (frees the A100)
import os, time, zipfile, shutil, glob
from google.colab import runtime
CK = f'{WORK}/checkpoints/sft_final.pt'
while not os.path.exists(CK):                      # waits here if training is still running
    print(time.strftime('%H:%M'), 'waiting for', CK, flush=True); time.sleep(300)
ZIP = f'{BASE}/{RUN}_model.zip'
items = [CK, f'{WORK}/tokenizer.json', f'{WORK}/plan.json', LM] + glob.glob(f'{WORK}/log_*.csv') + [f'{WORK}/train.log']
with zipfile.ZipFile('/content/model.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in items:
        if os.path.exists(f): z.write(f, os.path.basename(f))
assert zipfile.ZipFile('/content/model.zip').testzip() is None, 'zip is corrupt'
shutil.copy('/content/model.zip', ZIP)
ok = os.path.getsize(ZIP) == os.path.getsize('/content/model.zip') and zipfile.ZipFile(ZIP).testzip() is None
print('zip saved to Drive:', ZIP, f'{os.path.getsize(ZIP) / 1e6:.0f} MB', 'verified' if ok else 'VERIFY FAILED')
# Download the zip from Google Drive (drive.google.com > My Drive), not from this runtime: a browser download from
# Colab streams through the notebook connection (slow for ~260 MB) and dies when the runtime disconnects.
if ok:
    from google.colab import drive
    drive.flush_and_unmount()                        # make sure Drive has finished uploading before the runtime goes
    print('Drive synced; disconnecting runtime'); runtime.unassign()